<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/01_gpt2_from_the_hub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 1: GPT-2 from the 🤗 Hub

**Hugging Face** is the "GitHub of machine learning". Its **Hub** ([huggingface.co](https://huggingface.co)) hosts millions of models and datasets that anyone can download, and it also makes the Python libraries most people use to work with them:

| library | what it does |
|---|---|
| `transformers` | model classes (GPT-2, BERT, Llama, ...), downloading and running models |
| `huggingface_hub` | talking to the Hub: logging in, uploading, downloading |
| `datasets` | loading, filtering and sharing datasets (notebook 3) |
| `tokenizers` | fast tokenizers, used inside `transformers` |

Last week we downloaded GPT-2 with one line and opened it up. Today we look at that one line more closely.

**What you will do:**
- look at a model repository on the Hub, and see what `from_pretrained` downloads
- generate text with GPT-2, and collect the generated texts into a table
- log in to Hugging Face with an access token (`HF_TOKEN`)
- upload the generated texts to your account

**Contents:** 0. Setup · 1. A model on the Hub · 2. Downloading a model · 3. Generating text · 4. Generating data · 5. Logging in with `HF_TOKEN` · 6. Uploading to the Hub · 7. Summary

**Before you start:** *Runtime → Change runtime type → T4 GPU* (it also runs on a CPU, just slower). You need your Hugging Face account from Assignment 02.

---
## 0. Setup

`transformers` and `huggingface_hub` are already installed on Colab.

In [ ]:
import torch
import transformers

transformers.logging.set_verbosity_error()      # hide transformers' warnings, they are harmless in this notebook
print("transformers", transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

---
## 1. A model on the Hub

Every model on the Hub lives in a **repository** called `owner/name`, which is a git repository with a web page. GPT-2 is [openai-community/gpt2](https://huggingface.co/openai-community/gpt2) (the short name `gpt2` that we used last week still works). Open the page and look at:
- the **model card**: a description of the model, its training data and its limits
- **Files and versions**: the files in the repository
- the number of downloads, and *Use this model*, which shows the code to load it

With `HfApi` we can read the same information from Python, without downloading the model:

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
info = api.model_info("openai-community/gpt2", files_metadata=True)

print(f"{info.id}: {info.downloads:,} downloads last month, task: {info.pipeline_tag}")
total = 0
for f in info.siblings:
    size = f.size or 0
    total += size
    print(f"   {f.rfilename:35} {size / 1e6:8.1f} MB")
print(f"   {'TOTAL':35} {total / 1e6:8.1f} MB")

More than 6 GB! The same weights are stored many times, in formats for different libraries: `model.safetensors` and `pytorch_model.bin` for PyTorch, `tf_model.h5` for TensorFlow, `flax_model.msgpack` for JAX, `onnx/` and `.tflite` for phones and browsers, `rust_model.ot` for Rust. We need only a few of them.

---
## 2. Downloading a model

`from_pretrained` downloads exactly what it needs: the **config** (`config.json`: how many blocks, how wide, ...), the **weights** (`model.safetensors`) and, for the tokenizer, `tokenizer.json` and friends. Then it builds the model from the config and loads the weights into it.

`AutoModelForCausalLM` reads `config.json`, sees `"model_type": "gpt2"` and picks the right class (`GPT2LMHeadModel`) for us. *Causal LM* = a model that predicts the next token.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(type(model).__name__)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

*(On Colab you may see a warning about a missing `HF_TOKEN`. In section 5 we fix it.)*

**Look inside: where did it go?** Into a cache folder. The next `from_pretrained` of the same model is instant: nothing is downloaded again.

In [ ]:
from huggingface_hub import scan_cache_dir

for repo in scan_cache_dir().repos:
    files = sorted(f.file_name for revision in repo.revisions for f in revision.files)
    print(f"{repo.repo_id}: {repo.size_on_disk / 1e6:.0f} MB")
    print("   ", files)

About 550 MB instead of 6 GB: only the config, the PyTorch weights and the tokenizer files. We can also download a single file with `hf_hub_download`, e.g. the config:

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(model_id, "config.json")
print(path)
print(open(path).read())

The numbers from last week: `n_layer` 12, `n_head` 12, `n_embd` 768, `n_ctx` 1,024, `vocab_size` 50,257.

---
## 3. Generating text

The easiest way is a **pipeline**: it tokenizes the prompt, runs `generate`, and decodes the tokens back to text.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

torch.manual_seed(42)
result = generator("The capital of France is", max_new_tokens=30, do_sample=True, temperature=0.8)
print(result[0]["generated_text"])

Under the hood it is `model.generate()`. It takes token ids, and has many settings for how to pick the next token:

In [ ]:
ids = tokenizer("The capital of France is", return_tensors="pt").to(device)

settings = {
    "greedy (always the most likely token)": dict(do_sample=False),
    "sampling, temperature 0.7":             dict(do_sample=True, temperature=0.7),
    "sampling, temperature 1.5":             dict(do_sample=True, temperature=1.5),
    "sampling from the top 50 tokens only":  dict(do_sample=True, top_k=50),
}
for name, kwargs in settings.items():
    torch.manual_seed(0)
    out = model.generate(**ids, max_new_tokens=25, pad_token_id=tokenizer.eos_token_id, **kwargs)
    print(f"--- {name}")
    print(tokenizer.decode(out[0]))

- **greedy**: always the same text, and it often starts repeating itself
- **temperature**: as last week, low = safe and boring, high = wild
- **top_k**: draw only from the 50 most likely tokens, so the model never picks something absurd

**Try it:** Your own prompts. Remember that GPT-2 is not a chatbot: it continues text, it does not answer questions.

---
## 4. Generating data

A language model can produce as much text as you want. Such **synthetic data** is used, e.g., to train or test smaller models. (Today, people use much stronger models for this, see notebook 2. GPT-2's texts are mostly nonsense, but the workflow is the same.)

`num_return_sequences` gives several continuations of one prompt at once. We collect them into a `pandas` table:

In [ ]:
import pandas as pd

prompts = ["This movie was", "The best thing about Prague is", "My favourite food is"]

torch.manual_seed(1)
rows = []
for prompt in prompts:
    for result in generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, num_return_sequences=5):
        rows.append({"prompt": prompt, "text": result["generated_text"]})

df = pd.DataFrame(rows)
df

**Try it:** More prompts, more texts per prompt, other temperatures.

---
## 5. Logging in with `HF_TOKEN`

Downloading public models works without logging in. You need to log in to:
- **upload** anything (models, datasets) to your account
- download **private** repositories and **gated** models: e.g. Llama or Gemma, where you first accept the license on the model page
- get higher download limits (that is what the `HF_TOKEN` warning was about)

On Colab or a server you cannot click "log in" in a browser. Instead you use an **access token**: a long random password that starts with `hf_`.

**1. Create a token:** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → *Create new token* → *Write* → copy it. (If you kept the token from Assignment 02, you can reuse it.)

**2. Store it in Colab Secrets:**
- click the 🔑 icon in the left sidebar → *Add new secret*
- Name: `HF_TOKEN`, Value: your token
- switch on *Notebook access*

The `huggingface_hub` library finds the `HF_TOKEN` secret **automatically**, in every notebook you open. You never have to paste the token again. (Outside Colab, the same works with an environment variable `HF_TOKEN`.)

⚠️ **A token is a password.** Never write it into a notebook cell: notebooks get shared and pushed to GitHub, and bots scan GitHub for tokens within minutes. If a token leaks, delete it on the tokens page.

Let's check who we are:

In [ ]:
from huggingface_hub import whoami

user = whoami()["name"]
print("logged in as", user)

*(Colab may ask whether this notebook can access the `HF_TOKEN` secret: click *Grant access*. If you get an error, the secret is missing. As a fallback, run `from huggingface_hub import login; login()` and paste the token, as in Assignment 02.)*

---
## 6. Uploading to the Hub

Let's upload our generated texts as a **dataset** repository. `Dataset.from_pandas` turns the table into a Hugging Face dataset (more about datasets in notebook 3), and `push_to_hub` creates the repository and uploads it:

In [ ]:
from datasets import Dataset

generated = Dataset.from_pandas(df)
generated.push_to_hub(f"{user}/gpt2-generated")
print(f"https://huggingface.co/datasets/{user}/gpt2-generated")

Open the link: the dataset has its own page, with a viewer that shows the table. Anyone can now load it with `load_dataset("your-username/gpt2-generated")`.

**Models are uploaded the same way.** If you train or fine-tune a model (e.g. last week's mini GPT, or the BERT classifier in notebook 4), you upload it with:

```python
model.push_to_hub(f"{user}/my-model")          # config.json + model.safetensors
tokenizer.push_to_hub(f"{user}/my-model")      # tokenizer.json + ...
```

and then anyone can use it with `AutoModelForCausalLM.from_pretrained("your-username/my-model")` or with `pipeline`. Add `private=True` if you do not want others to see it. For any other file there is `HfApi().upload_file(...)`, which you used in Assignment 02.

---
## 7. Summary

| step | code |
|---|---|
| information about a model | `HfApi().model_info("owner/name")` |
| downloading a model | `AutoModelForCausalLM.from_pretrained(...)`, `AutoTokenizer.from_pretrained(...)` |
| downloading one file | `hf_hub_download(repo, filename)` |
| generating | `pipeline("text-generation", ...)`, `model.generate(..., do_sample, temperature, top_k)` |
| logging in | the `HF_TOKEN` secret in Colab (or `login()`) |
| uploading | `dataset.push_to_hub(...)`, `model.push_to_hub(...)`, `HfApi().upload_file(...)` |

**New words:** Hugging Face Hub, repository, model card, cache, config, safetensors, greedy decoding, sampling, top-k, synthetic data, access token, Colab Secrets, gated model.

### Exercises
1. Find a bigger GPT-2 on the Hub (`openai-community/gpt2-medium`, `-large`, `-xl`). How big is it? Does it write better texts?
2. Find a small modern model on the Hub (e.g. search for *SmolLM* or *Qwen*) and generate with it. Which settings does its model card recommend?
3. Make your `gpt2-generated` dataset private (*Settings* on its page). Can your neighbour still load it?

### Further reading
- [Hugging Face Hub documentation](https://huggingface.co/docs/hub): repositories, model cards, security tokens
- [How to generate text](https://huggingface.co/blog/how-to-generate): greedy search, sampling, top-k and top-p explained